# Practice Lab: Advanced Retrieval (Lesson 8.4)

Module 8 · 8 exercises · Hybrid search + reranking + HyDE + routing + CRAG + evaluation

Exercises 1, 2, 4, 5, 7, 8 run locally (numpy only).


# Lesson 8.4: Advanced Retrieval — Beyond Naive Cosine

8 exercises: 2E + 3M + 3C

Exercises 1, 2, 4, 5, 7, 8 run **locally** (numpy). Ex 3, 6 are design/architecture.


In [ ]:
import numpy as np
import hashlib, math
from collections import Counter


---
## Exercise 1 (Easy): Hybrid Search + RRF


In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
import math, hashlib
import numpy as np
from collections import Counter

class BM25:
    def __init__(self, docs, k1=1.5, b=0.75):
        self.docs = [d.lower().split() for d in docs]
        self.k1, self.b = k1, b
        self.avg_dl = sum(len(d) for d in self.docs) / len(self.docs)
        self.df = Counter()
        for d in self.docs:
            for w in set(d): self.df[w] += 1
        self.N = len(self.docs)
    def rank(self, query, k=5):
        q = query.lower().split(); scores = []
        for doc in self.docs:
            s, tf = 0, Counter(doc)
            for w in q:
                if w not in self.df: continue
                idf = math.log((self.N-self.df[w]+0.5)/(self.df[w]+0.5)+1)
                s += idf * tf.get(w,0)*(self.k1+1)/(tf.get(w,0)+self.k1*(1-self.b+self.b*len(doc)/self.avg_dl))
            scores.append(s)
        return sorted(enumerate(scores), key=lambda x: -x[1])[:k]

def fake_embed(text, dim=64):
    h = hashlib.sha256(text.lower().encode()).digest()
    vec = np.array([b/255.0 for b in h[:dim]])
    kw = {"refund":0,"money":1,"back":2,"nts":3,"4021":4,"error":5,"genai":6,"course":7,"prerequisite":8,"python":9}
    for w, i in kw.items():
        if w in text.lower() and i < dim: vec[i] += 0.5
    return vec / np.linalg.norm(vec)

def dense_rank(query, docs, k=5):
    qe = fake_embed(query)
    return sorted([(i, np.dot(qe, fake_embed(d))/(np.linalg.norm(qe)*np.linalg.norm(fake_embed(d)))) for i, d in enumerate(docs)], key=lambda x: -x[1])[:k]

def rrf(rankings, k=60):
    scores = {}
    for ranking in rankings:
        for rank, (idx, _) in enumerate(ranking):
            scores[idx] = scores.get(idx, 0) + 1.0/(k+rank+1)
    return sorted(scores.items(), key=lambda x: -x[1])

docs = ["Refund policy: full refund within 7 days of purchase",
        "Error code NTS-4021: authentication token expired",
        "The GenAI course covers transformers RAG fine-tuning",
        "Return and refund requests via support email",
        "NTS-4021 troubleshooting: clear cache re-authenticate",
        "Prerequisites: basic Python and high school math",
        "Course costs 14999 rupees with lifetime access",
        "Live classes daily 7 PM IST on YouTube recordings"]
bm25 = BM25(docs)

for query, qtype in [("how to get my money back","semantic"),("NTS-4021","exact"),("GenAI prerequisites price","mixed")]:
    br = bm25.rank(query); dr = dense_rank(query, docs); hr = rrf([br, dr])[:3]
    print(f"  Q: {query} ({qtype})")
    print(f"    BM25:   [{br[0][0]}] {docs[br[0][0]][:45]}...")
    print(f"    Dense:  [{dr[0][0]}] {docs[dr[0][0]][:45]}...")
    print(f"    Hybrid: [{hr[0][0]}] {docs[hr[0][0]][:45]}...")

print(f"\nRRF: score=sum(1/(60+rank)). Impact: +26% NDCG")


</details>


---
## Exercise 2 (Easy): Cross-Encoder Reranking


In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
import numpy as np, hashlib

def fake_embed(t, dim=64):
    h = hashlib.sha256(t.lower().encode()).digest()
    vec = np.array([b/255.0 for b in h[:dim]])
    kw = {"prerequisite":0,"python":1,"math":2,"basic":3,"course":4,"price":5,"refund":6}
    for w, i in kw.items():
        if w in t.lower() and i < dim: vec[i] += 0.5
    return vec / np.linalg.norm(vec)

def cross_encoder_score(q, d):
    q_w = set(q.lower().split()); d_w = set(d.lower().split())
    overlap = len(q_w & d_w)
    early = len(q_w & set(d.lower().split()[:8])) * 0.3
    return overlap + early

docs = ["GenAI course costs 14999 rupees lifetime access",
        "Prerequisites: basic Python programming and math",
        "Live classes daily at 7 PM IST on YouTube",
        "Students should know Python basics loops functions",
        "Refund policy allows full refunds within 7 days",
        "Certificate awarded on 85 percent completion",
        "EMI via Razorpay starting 2500 per month",
        "14 modules Python to production GCP deployment"]
query = "What are the prerequisites for the GenAI course?"

qe = fake_embed(query)
initial = sorted([(i, np.dot(qe, fake_embed(d))/(np.linalg.norm(qe)*np.linalg.norm(fake_embed(d)))) for i, d in enumerate(docs)], key=lambda x: -x[1])
reranked = sorted([(i, cross_encoder_score(query, docs[i])) for i, _ in initial[:6]], key=lambda x: -x[1])

print(f"Query: {query}\n")
print("Before (bi-encoder):")
for r, (i, s) in enumerate(initial[:5]):
    m = " <<<" if "prerequisit" in docs[i].lower() else ""
    print(f"  [{r+1}] {docs[i][:45]}... ({s:.3f}){m}")
print("\nAfter (cross-encoder reranking):")
for r, (i, s) in enumerate(reranked[:5]):
    m = " <<<" if "prerequisit" in docs[i].lower() else ""
    print(f"  [{r+1}] {docs[i][:45]}... ({s:.1f}){m}")

print(f"\nImpact: +15-40% accuracy, Hit@1 63%->83%")
print(f"Rerankers: FlashRank(4MB), BGE(free), Cohere($2/1K)")


</details>


---
## Exercise 4 (Medium): Query Router


In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
class QueryRouter:
    ROUTES = {
        "vector_store": {"keywords": ["refund","policy","price","cost","prerequisite","course","module","certificate","emi","schedule"]},
        "web_search": {"keywords": ["latest","news","today","current","2026","recent","update"]},
        "direct_llm": {"keywords": ["what is","explain","define","how does","difference between"]},
        "multi_hop": {"keywords": ["compare","and also","both","relationship between","summarize all"]},
    }
    def classify(self, query):
        q = query.lower()
        scores = {r: sum(1 for kw in c["keywords"] if kw in q) for r, c in self.ROUTES.items()}
        best = max(scores, key=scores.get)
        return best if scores[best] > 0 else "vector_store"

router = QueryRouter()
queries = ["What is the refund policy?", "Latest GPT-5 news?", "What is a transformer?",
           "Compare refund policy with competitor", "Course cost?", "What happened in AI today?",
           "Explain attention mechanism", "Summarize all modules and prerequisites",
           "Live class schedule?", "Current AI regulation?", "Define gradient descent",
           "How does EMI affect enrollment?"]

counts = {}
for q in queries:
    r = router.classify(q)
    counts[r] = counts.get(r, 0) + 1
    print(f"  [{r:<14}] {q[:40]}...")

print(f"\nDistribution:")
for r, c in sorted(counts.items(), key=lambda x: -x[1]):
    print(f"  {r:<14} {c} ({c/len(queries)*100:.0f}%) {'█'*int(c/len(queries)*20)}")

print(f"\nTaxonomy: I=direct, II=decompose, III=step-back, IV=full")
print(f"Impact: +18-25% accuracy with <1ms latency")


</details>


---
## Exercise 5 (Medium): CRAG Implementation


In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
class CRAGEvaluator:
    def evaluate(self, query, doc):
        qw = set(query.lower().split()); dw = set(doc.lower().split())
        return round(len(qw & dw) / max(len(qw), 1), 3)
    def route(self, query, doc):
        s = self.evaluate(query, doc)
        if s >= 0.4: return "correct", s
        elif s >= 0.2: return "ambiguous", s
        else: return "incorrect", s
    def process(self, query, doc):
        action, score = self.route(query, doc)
        if action == "correct": return {"action": "GENERATE", "source": "vector_store", "score": score}
        elif action == "incorrect": return {"action": "WEB SEARCH", "source": "web_search", "score": score}
        else: return {"action": "COMBINE both", "source": "both", "score": score}

crag = CRAGEvaluator()
cases = [
    ("What is the refund policy?", "Refund policy: full refund within 7 days"),
    ("What is the refund policy?", "Live classes held daily at 7 PM IST"),
    ("Latest GPT-5 benchmarks?", "GenAI course covers transformers RAG"),
    ("Course price?", "Course costs 14999 rupees lifetime access EMI"),
    ("Python version needed?", "Prerequisites basic Python programming math"),
    ("Current AI regulation?", "Certificate awarded upon course completion"),
]
ok_with, ok_without = 0, 0
print("CRAG Pipeline:")
for q, doc in cases:
    r = crag.process(q, doc)
    icon = {"vector_store":"OK","web_search":"!!","both":"??"}[r["source"]]
    print(f"  [{icon}] {q[:35]}... -> {r['action']} (score={r['score']:.3f})")
    if r["score"] >= 0.3: ok_without += 1
    ok_with += 1  # CRAG always routes correctly

print(f"\nWithout CRAG: {ok_without}/{len(cases)} (always use retrieved)")
print(f"With CRAG:    routes bad retrieval to web search")
print(f"PopQA: 78.1% vs 51.4% vanilla (+26.7 pts)")


</details>


---
## Exercise 7 (Challenge): RAGAS Evaluation Pipeline


In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
class RAGAS:
    @staticmethod
    def faithfulness(ans, ctx):
        claims = [c.strip() for c in ans.split(".") if len(c.strip()) > 10]
        if not claims: return 1.0, []
        sup, det = 0, []
        for cl in claims:
            cw = set(cl.lower().split()); xw = set(ctx.lower().split())
            ok = len(cw & xw)/max(len(cw),1) > 0.3
            sup += int(ok); det.append((cl[:35], ok))
        return sup/len(claims), det
    @staticmethod
    def ctx_prec(retrieved, relevant):
        if not relevant: return 1.0
        prec, found = [], 0
        for k, idx in enumerate(retrieved):
            if idx in relevant: found += 1; prec.append(found/(k+1))
        return sum(prec)/len(relevant) if prec else 0.0

ev = RAGAS(); TH = 0.7
cases = [
    {"q":"Refund?", "ctx":"Refund: full refund 7 days. 50% 8-30 days.", "ans":"Full refund 7 days, 50 percent 8-30 days.", "ret":[0,3,5], "rel":[0], "lbl":"Faithful"},
    {"q":"Refund?", "ctx":"Refund: full refund 7 days.", "ans":"Full refund 14 days and 75% after that.", "ret":[0,3,5], "rel":[0], "lbl":"Hallucinated"},
    {"q":"Price?", "ctx":"Course 14999 rupees lifetime access.", "ans":"Course costs 14999 rupees lifetime access.", "ret":[2,0,4], "rel":[2], "lbl":"Faithful"},
    {"q":"Live class?", "ctx":"Live classes daily 7 PM IST YouTube.", "ans":"Classes at 7 PM IST daily on YouTube.", "ret":[3,1,0], "rel":[3], "lbl":"Faithful"},
    {"q":"Prerequisites?", "ctx":"Prerequisites: basic Python and math.", "ans":"Need Python and math. Also Java and Rust.", "ret":[4,2,1], "rel":[4], "lbl":"Partial"},
    {"q":"EMI?", "ctx":"EMI Razorpay 2500 per month.", "ans":"EMI available Razorpay 2500 per month.", "ret":[5,0,2], "rel":[5], "lbl":"Faithful"},
]
print("RAGAS Evaluation:")
all_f = []
for c in cases:
    f, d = ev.faithfulness(c["ans"], c["ctx"])
    p = ev.ctx_prec(c["ret"], c["rel"])
    all_f.append(f)
    st = "PASS" if f >= TH else "FAIL"
    print(f"  [{st}] {c['q']} ({c['lbl']}): faith={f:.2f} prec={p:.2f}")
    for cl, ok in d: print(f"    [{'OK' if ok else '!!'}] {cl}...")

passed = sum(1 for f in all_f if f >= TH)
print(f"\nSummary: {passed}/{len(cases)} passed, avg={sum(all_f)/len(all_f):.2f}")
print(f"4-Layer Stack: golden tests -> CI/CD DeepEval -> A/B -> monitoring")


</details>


---
## Exercise 8 (Challenge): Agentic RAG


In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
class AgenticRAG:
    MAX_CYCLES = 3
    def __init__(self, kb): self.kb = kb
    def _score(self, q, d):
        qw = set(q.lower().split()); dw = set(d.lower().split())
        return len(qw & dw) / max(len(qw), 1)
    def run(self, query):
        q = query; log = []
        for cycle in range(self.MAX_CYCLES):
            best = max(self.kb, key=lambda d: self._score(q, d))
            score = self._score(q, best)
            action = "generate" if score > 0.4 else "rewrite" if score > 0.2 else "web_search"
            log.append({"cycle": cycle+1, "score": round(score,2), "action": action, "doc": best[:40]})
            if action != "rewrite": break
            q = query + " details " + " ".join(query.split()[:3])
        return log

kb = ["Refund policy full refund 7 days 50 percent 8-30 days",
      "GenAI course 14999 rupees lifetime access certificate",
      "Live classes daily 7 PM IST YouTube recordings",
      "Prerequisites basic Python math no ML needed",
      "EMI available Razorpay 2500 rupees per month"]

agent = AgenticRAG(kb)
for q in ["What is the refund policy?", "Latest GPT-5 benchmarks?", "Course pricing EMI?"]:
    log = agent.run(q)
    print(f"  Q: {q}")
    for s in log: print(f"    Cycle {s['cycle']}: {s['score']:.2f} -> {s['action'].upper()}")

print(f"\n7 Architectures: Naive, Advanced, Self-RAG, CRAG, Adaptive, Agentic, GraphRAG")
print(f"Start: CRAG (plug-and-play, +26.7 pts). Cap agents at 3 cycles.")


</details>


---
## Exercise 3 (Medium): Query Transformation Pipeline
Architecture/design. See practice-lab-8_4.html.


In [ ]:
# Exercise 3: Query Transformation Pipeline
pass


---
## Exercise 6 (Challenge): Contextual Retrieval
Architecture/design. See practice-lab-8_4.html.


In [ ]:
# Exercise 6: Contextual Retrieval
pass
